In [53]:
import sqlite3
import pandas as pd

DB_PATH = "/Users/darraghdonnelly/dev/Database/pacerDB.db"
USER_ID = 1  # allow predictions for specific user

# connect to sqlite db and read user_run table
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query("SELECT distance, time, user_id FROM user_runs", conn)

if USER_ID is not None:
    df = df[df["user_id"] == USER_ID]

df = df.dropna(subset=["distance", "time"])
df.head()

,distance,time,user_id
0,5.89,34.0,1
1,5.84,32.0,1
2,4.24,21.8,1
3,5.89,34.0,1
4,5.84,32.0,1


In [54]:
from sklearn.linear_model import LinearRegression

if len(df) < 2:
    raise ValueError("Need at least 2 runs to train model.")

X = df[["distance"]]
y = df["time"]

model = LinearRegression()
model.fit(X, y)

print(f"samples: {len(df)}")
print(f"coef: {model.coef_[0]:.4f}")
print(f"intercept: {model.intercept_:.4f}")


samples: 28
coef: 5.2865
intercept: -1.6128


In [57]:
# basic prediction for a given distance using linear regression model
def predict_time(input_distance):
    return float(model.predict([[input_distance]])[0])

predict_time(5)

/Users/darraghdonnelly/dev/fyp-running-app/ml/.mlvenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


24.81945632483137

In [56]:
import joblib
from pathlib import Path

MODEL_PATH = Path("/Users/darraghdonnelly/dev/fyp-running-app/backend/ml_models") / f"user_{USER_ID}_linreg.joblib"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)

['/Users/darraghdonnelly/dev/fyp-running-app/backend/ml_models/user_1_linreg.joblib']